<a href="https://colab.research.google.com/github/AfrinHossai/cs171-police-call-forecasting/blob/main/03_baseline_and_tree_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [42]:
# Task 2.1: Creating simple baseline models

# Step 1: Obtaining the train dataset
import pandas as pd
import numpy as np

processed_folder = "/content/drive/MyDrive/CS171_Police_Call_Project/data_processed"

train_scaled_df = pd.read_csv(
    f"{processed_folder}/train_scaled.csv"
)

test_scaled_df = pd.read_csv(
    f"{processed_folder}/test_scaled.csv"
)

# Step 2: Creating the baseline model : a simple mean predictor
total_calls = train_scaled_df['NEXT_HOUR_CALLS'].sum()
num_hours = len(train_scaled_df)

mean_hourly_call_count = total_calls / num_hours
print(f"Mean hourly count: Approximately {mean_hourly_call_count:.5} calls are received every hour.")

# Step 3: Creating a new column that represents a persistence forecast (uses the previous hour's count)
train_scaled_df['PERSISTENCE_PREDICTION'] = train_scaled_df['NEXT_HOUR_CALLS'].shift(1)
test_scaled_df['PERSISTENCE_PREDICTION'] = test_scaled_df['NEXT_HOUR_CALLS'].shift(1)

train_scaled_df.head()

Mean hourly count: Approximately 30.775 calls are received every hour.


,HOUR,NEXT_TIMESTAMP,TARGET_HOUR,TARGET_DAY_OF_WEEK,TARGET_MONTH,TARGET_IS_WEEKEND,HOUR_SIN,HOUR_COS,DAY_OF_WEEK_SIN,DAY_OF_WEEK_COS,...,CURRENT_CALLTYPE_THEFT_COUNT,CURRENT_CALLTYPE_RECKLESS_DRIVING_COUNT,CURRENT_CALLTYPE_VEHICLE_ACCIDENT_PROPERTY_DAMAGE_COUNT,CURRENT_CALLTYPE_PEDESTRIAN_STOP_COUNT,CURRENT_CALLTYPE_TRAFFIC_HAZARD_COUNT,CURRENT_CALLTYPE_MEET_THE_CITIZEN_COUNT,CURRENT_CALLTYPE_RECOVERED_STOLEN_VEHICLE_COUNT,CURRENT_CALLTYPE_OTHER_COUNT,NEXT_HOUR_CALLS,PERSISTENCE_PREDICTION
0,2024-01-07 23:00:00,2024-01-08 00:00:00,-1.661609,-1.49438,-1.572991,-0.630874,0.000095,1.414426,-0.00194,1.409718,...,-0.686278,0.392031,0.533891,0.66914,-0.508273,-0.641627,2.318661,-0.956892,34,NaN
1,2024-01-08 00:00:00,2024-01-08 01:00:00,-1.517137,-1.49438,-1.572991,-0.630874,0.366108,1.366237,-0.00194,1.409718,...,-0.686278,1.476900,-0.669392,0.66914,-0.508273,-0.641627,-0.577590,-0.715549,21,34.0
2,2024-01-08 01:00:00,2024-01-08 02:00:00,-1.372666,-1.49438,-1.572991,-0.630874,0.707178,1.224951,-0.00194,1.409718,...,-0.686278,0.392031,-0.669392,0.66914,-0.508273,-0.641627,-0.577590,-1.439577,19,21.0
3,2024-01-08 02:00:00,2024-01-08 03:00:00,-1.228194,-1.49438,-1.572991,-0.630874,1.000062,1.000199,-0.00194,1.409718,...,-0.686278,-0.692839,-0.669392,-0.59257,0.579574,-0.641627,2.318661,-0.715549,12,19.0
4,2024-01-08 03:00:00,2024-01-08 04:00:00,-1.083722,-1.49438,-1.572991,-0.630874,1.224799,0.707296,-0.00194,1.409718,...,-0.686278,-0.692839,-0.669392,0.66914,-0.508273,-0.641627,-0.577590,-1.439577,16,12.0


In [85]:
# Task 2.2: Train a linear model
from sklearn.linear_model import LinearRegression
from time import perf_counter

X_train_r = train_scaled_df.drop(columns=['HOUR', 'NEXT_TIMESTAMP', 'NEXT_HOUR_CALLS', 'PERSISTENCE_PREDICTION'])
y_train_r = train_scaled_df['NEXT_HOUR_CALLS']

X_test_r = test_scaled_df.drop(columns=['HOUR', 'NEXT_TIMESTAMP', 'NEXT_HOUR_CALLS', 'PERSISTENCE_PREDICTION'])
y_test_r = test_scaled_df['NEXT_HOUR_CALLS']

# Step 1: Model fitting

lr_st = perf_counter()
lr = LinearRegression().fit(X_train_r, y_train_r)
lr_et = perf_counter()

lr_training_time = lr_et - lr_st

y_intercept = round(lr.intercept_, 3)
coefficients = lr.coef_
features = list(X_train_r.columns.values)

y = f"{y_intercept}"

for coef, feature in zip(coefficients, features):
  y += f" + {round(coef, 3)}*{feature}"

print(f"Learned y-intercept: {y_intercept}")
print(f"Learned Equation: {y}")

print()

# Step 2: Predicting on the test dataset

predictions = lr.predict(X_test_r)

p_10 = predictions[0:10]
a_10 = y_test_r[0:10]

print("Pred  Actual")
for p, a in zip(p_10, a_10):
    print(round(p, 3), a)

print()

# Step 3: Compute Loss Functions
n = p_10.size
subtractions = (np.subtract(p_10, a_10))

# (a) MSE:
squares = (np.square(subtractions))
mse = (1 / n) * (np.sum(squares))


# (b) MAE:
abs_vals = np.absolute(subtractions)
mae = (1 / n) * np.sum(abs_vals)

# (c) Huber Loss:
delta = 20.0
contributions = np.where(abs_vals <= delta, 0.5 * abs_vals ** 2, delta * (abs_vals - 0.5 * delta))
huber_loss = np.sum(contributions) / n

print(f"Training time: {lr_training_time:.3} seconds")
print(f"Mean Squared Error: {mse:.4}")
print(f"Mean Absolute Error: {mae:.4}")
print(f"Huber Loss: {huber_loss:.4}")


print()


Learned y-intercept: 30.775
Learned Equation: 30.775 + 2.338*TARGET_HOUR + -0.467*TARGET_DAY_OF_WEEK + -0.013*TARGET_MONTH + 0.063*TARGET_IS_WEEKEND + -1.027*HOUR_SIN + -1.437*HOUR_COS + -0.431*DAY_OF_WEEK_SIN + -0.655*DAY_OF_WEEK_COS + -0.028*MONTH_SIN + -0.245*MONTH_COS + 0.741*LAG_1H + 1.205*LAG_2H + 0.235*LAG_3H + 1.373*LAG_24H + 0.329*LAG_48H + 2.448*LAG_168H + 0.819*ROLLING_MEAN_3H + -1.634*ROLLING_MEAN_6H + 0.058*ROLLING_MEAN_24H + 0.065*ROLLING_MEAN_168H + -0.246*CURRENT_PRIORITY_1_COUNT + -0.234*CURRENT_PRIORITY_2_COUNT + -0.076*CURRENT_PRIORITY_3_COUNT + 0.743*CURRENT_PRIORITY_4_COUNT + 0.145*CURRENT_PRIORITY_5_COUNT + 1.832*CURRENT_PRIORITY_6_COUNT + -0.002*CURRENT_PRIORITY_OTHER_COUNT + 0.116*CURRENT_CALLTYPE_VEHICLE_STOP_COUNT + 0.315*CURRENT_CALLTYPE_DISTURBANCE_COUNT + 0.188*CURRENT_CALLTYPE_WELFARE_CHECK_COUNT + 0.136*CURRENT_CALLTYPE_ALARM_AUDIBLE_COUNT + 0.149*CURRENT_CALLTYPE_PARKING_VIOLATION_COUNT + 1.382*CURRENT_CALLTYPE_DISTURBANCE_MUSIC_COUNT + 0.059*CURRENT_CAL

In [88]:
# Task 2.3: Train a decision tree

from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Step 1: Basic fitting
dt_regressor = DecisionTreeRegressor(max_depth=5, random_state=42)

dt_st = perf_counter()
dt_regressor.fit(X_train_r, y_train_r)
dt_et = perf_counter()
dt_training_time = dt_et - dt_st

y_pred = dt_regressor.predict(X_test_r)
y_train_pred = dt_regressor.predict(X_train_r)

mse = mean_squared_error(y_test_r, y_pred)

print(f"Training Time (Before hyperparameter tuning): {dt_training_time:.3} seconds")
print(f"Decision Tree Mean Squared Error (Before hyperparamter tuning): {mse:.3f}")

print()

# Step 2: Tuning max_depth parameter
max_depth_list = [1, 3, 5, 7, 9, 15, 25]

for depth in max_depth_list:
  regressor = DecisionTreeRegressor(max_depth=depth, random_state=42)
  regressor.fit(X_train_r, y_train_r)

  y_prediction = regressor.predict(X_test_r)

  mse_val = mean_squared_error(y_test_r, y_prediction)

  print(f"Max depth : {depth}")
  print(f"Mean Squared Error: {mse_val:.3}")

  print("--------------------------------------")

print()

# Step 3: Tuning the min # of samples a leaf node requires
leaf_sizes_list = [1, 5, 10, 15, 25, 50, 75, 100, 125]

for leaf_size in leaf_sizes_list:
  regressor = DecisionTreeRegressor(max_depth=5, random_state=42, min_samples_leaf=leaf_size)
  regressor.fit(X_train_r, y_train_r)

  y_prediction = regressor.predict(X_test_r)

  mse_val = mean_squared_error(y_test_r, y_prediction)

  print(f"Min # of samples : {leaf_size}")
  print(f"Mean Squared Error: {mse_val:.3}")
  print("---------------------------------------")


Training Time (Before hyperparameter tuning): 0.113 seconds
Decision Tree Mean Squared Error (Before hyperparamter tuning): 83.605

Max depth : 1
Mean Squared Error: 1.68e+02
--------------------------------------
Max depth : 3
Mean Squared Error: 1.1e+02
--------------------------------------
Max depth : 5
Mean Squared Error: 83.6
--------------------------------------
Max depth : 7
Mean Squared Error: 78.5
--------------------------------------
Max depth : 9
Mean Squared Error: 88.6
--------------------------------------
Max depth : 15
Mean Squared Error: 1.07e+02
--------------------------------------
Max depth : 25
Mean Squared Error: 1.18e+02
--------------------------------------

Min # of samples : 1
Mean Squared Error: 83.6
---------------------------------------
Min # of samples : 5
Mean Squared Error: 84.0
---------------------------------------
Min # of samples : 10
Mean Squared Error: 84.2
---------------------------------------
Min # of samples : 15
Mean Squared Error: 84.

In [ ]:
# # Task 2.3.1: Visual Plots

In [89]:
# Task 2.4: Train a random forest

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

# Step 1: Basic fitting
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)

rf_st = perf_counter()
rf_regressor.fit(X_train_r, y_train_r)
rf_et = perf_counter()
rf_training_time = rf_et - rf_st

rf_y_pred = rf_regressor.predict(X_test_r)

rf_mae = mean_absolute_error(y_test_r, rf_y_pred)
rf_r2 = r2_score(y_test_r, rf_y_pred)
rf_rmse = root_mean_squared_error(y_test_r, rf_y_pred)

print(f"Training Time (Before hyperparameter tuning): {rf_training_time:.3} seconds")
print(f"Random Forest Mean Absolute Error: {rf_mae:.3f}")
print(f"Random Forest Root Mean Squared Error {rf_rmse:.3f}")
print(f"Random Forest R^2 Score: {rf_r2:.3f}")
print()

# Step 2: Tuning n_estimators parameter
n_estimators_list = [1, 5, 10, 25, 50, 75, 100, 125]

for num in n_estimators_list:
  regressor = RandomForestRegressor(n_estimators=num, max_depth=5, random_state=42)
  regressor.fit(X_train_r, y_train_r)

  y_prediction = regressor.predict(X_test_r)

  mae_val = mean_absolute_error(y_test_r, y_prediction)
  r2_val = r2_score(y_test_r, y_prediction)
  rmse_val = root_mean_squared_error(y_test_r, y_prediction)

  print(f"{num} estimator(s):")
  print(f"MAE: {mae_val:.3f}")
  print(f"R^2: {r2_val:.3f}")
  print(f"RMSE: {rmse_val:.3f}")
  print("-----------------------------------------------------------")


print()

# Step 2: Tuning max_features parameter
max_features_list = [5, 10, 15, 20, 25, 50, 75]

for num in max_features_list:
  regressor = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, max_features=num)

  regressor.fit(X_train_r, y_train_r)

  y_prediction = regressor.predict(X_test_r)

  mae_val = mean_absolute_error(y_test_r, y_prediction)
  r2_val = r2_score(y_test_r, y_prediction)
  rmse_val = root_mean_squared_error(y_test_r, y_prediction)

  print(f"{num} max_features:")
  print(f"MAE: {mae_val:.3f}")
  print(f"R^2: {r2_val:.3f}")
  print(f"RMSE: {rmse_val:.3f}")
  print("--------------------------------------")



Training Time (Before hyperparameter tuning): 39.5 seconds
Random Forest Mean Absolute Error: 5.647
Random Forest Root Mean Squared Error 8.081
Random Forest R^2 Score: 0.715

1 estimator(s):
MAE: 6.454
R^2: 0.603
RMSE: 9.541
-----------------------------------------------------------
5 estimator(s):
MAE: 6.174
R^2: 0.641
RMSE: 9.071
-----------------------------------------------------------
10 estimator(s):
MAE: 6.029
R^2: 0.664
RMSE: 8.779
-----------------------------------------------------------
25 estimator(s):
MAE: 5.977
R^2: 0.673
RMSE: 8.666
-----------------------------------------------------------
50 estimator(s):
MAE: 5.966
R^2: 0.674
RMSE: 8.651
-----------------------------------------------------------
75 estimator(s):
MAE: 5.973
R^2: 0.672
RMSE: 8.675
-----------------------------------------------------------
100 estimator(s):
MAE: 5.967
R^2: 0.673
RMSE: 8.662
-----------------------------------------------------------
125 estimator(s):
MAE: 5.962
R^2: 0.674
RMSE: 8.